In [1]:
import numpy as np
from scipy.linalg import lu, solve_triangular

# ---- 创建矩阵和多个右端项 ----
A = np.array([[2, 1, 1],
              [4, -6, 0],
              [-2, 7, 2]], dtype=float)

b1 = np.array([1, 2, 3], dtype=float)   # 第一个右端项
b2 = np.array([4, 5, 6], dtype=float)   # 第二个右端项
b3 = np.array([7, 8, 9], dtype=float)   # 第三个右端项

# ---- LU 分解 ----
# P 是置换矩阵（处理行交换），L 是下三角，U 是上三角
P, L, U = lu(A)

print("===== LU 分解 =====")
print("A = P @ L @ U")
print("P = \n", P)
print("L = \n", L)
print("U = \n", U)

# 验证：P @ L @ U 是否等于 A
print(f"\n分解是否正确? {np.allclose(A, P @ L @ U)}")

# ---- 使用 LU 分解解多个右端项 ----
print("\n===== 用 LU 解多个右端项 =====")

def solve_with_lu(P, L, U, b):
    # 1. 先对右端项应用置换矩阵 P (因为 A = P L U, 所以 A x = b 变成 P L U x = b)
    # 等价于解 L U x = P^T b (因为 P 是正交矩阵，P^T = P^{-1})
    b_permuted = P.T @ b

    # 2. 前向替换: 解 L c = b_permuted
    c = solve_triangular(L, b_permuted, lower=True)

    # 3. 回代: 解 U x = c
    x = solve_triangular(U, c, lower=False)

    return x

# 分别解三个右端项
x1 = solve_with_lu(P, L, U, b1)
x2 = solve_with_lu(P, L, U, b2)
x3 = solve_with_lu(P, L, U, b3)

print(f"解 b1 = {b1}: x1 = {x1}")
print(f"解 b2 = {b2}: x2 = {x2}")
print(f"解 b3 = {b3}: x3 = {x3}")

# ---- 验证：直接和 LU 结果一致 ----
print("\n===== 验证 =====")
x1_direct = np.linalg.solve(A, b1)
print(f"直接解 x1: {x1_direct}")
print(f"差异: {np.linalg.norm(x1 - x1_direct):.2e}")

# ---- ⚠️ LU 分解的优势：只需要分解一次 ----
# 如果 n=1000，解一个右端项需要 ~1/3 n^3 次操作
# 分解一次需要 ~1/3 n^3 次操作，然后每个右端项只需 ~n^2 次操作
# 如果你有 100 个右端项，效率提升 ~100 倍！
print("\n===== 效率对比 =====")
n = 500
A_big = np.random.randn(n, n)
B_big = np.random.randn(n, 10)  # 10个右端项

# 方法1：用 solve 解10次
import time
start = time.time()
for i in range(10):
    x = np.linalg.solve(A_big, B_big[:, i])
time_direct = time.time() - start

# 方法2：LU分解 + 解10次
start = time.time()
P, L, U = lu(A_big)
for i in range(10):
    b = B_big[:, i]
    b_permuted = P.T @ b
    c = solve_triangular(L, b_permuted, lower=True)
    x = solve_triangular(U, c, lower=False)
time_lu = time.time() - start

print(f"直接解 10 次: {time_direct:.3f} 秒")
print(f"LU 分解 + 解 10 次: {time_lu:.3f} 秒")
print(f"LU 方法快了 {time_direct/time_lu:.1f} 倍")

===== LU 分解 =====
A = P @ L @ U
P = 
 [[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]
L = 
 [[ 1.   0.   0. ]
 [ 0.5  1.   0. ]
 [-0.5  1.   1. ]]
U = 
 [[ 4. -6.  0.]
 [ 0.  4.  1.]
 [ 0.  0.  1.]]

分解是否正确? True

===== 用 LU 解多个右端项 =====
解 b1 = [1. 2. 3.]: x1 = [-1. -1.  4.]
解 b2 = [4. 5. 6.]: x2 = [-0.8125 -1.375   7.    ]
解 b3 = [7. 8. 9.]: x3 = [-0.625 -1.75  10.   ]

===== 验证 =====
直接解 x1: [-1. -1.  4.]
差异: 0.00e+00

===== 效率对比 =====
直接解 10 次: 0.659 秒
LU 分解 + 解 10 次: 0.023 秒
LU 方法快了 29.0 倍
